# Simple – The Price Is Right 🛍️💰

Can an AI guess the price of an Amazon product just by reading its description?

In this notebook, we put **GPT-4o-mini** to the test. We'll see whether a language model can estimate what something costs — from a fancy kitchen gadget to a random USB cable — using nothing but the product description.

### What we'll do

1. **Load the data** – Start with a small sample of Amazon product listings data
2. **Set a baseline** – Our “lazy” model always guesses the **average price**  
3. **Ask the AI** – Let **GPT-4o-mini** read the description and guess the price  
4. **Judge the results** – Compare predictions using **Mean Absolute Error (MAE)**  
5. **Play with it yourself** – Build a **Gradio UI** where you can paste any product description and see what the AI thinks it costs

By the end, you'll know whether the AI is a **savvy online shopper**… or completely terrible at guessing prices. 🧠📦

In [ ]:
import re
import random
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_dataset
from tqdm import tqdm

load_dotenv(override=True)
client = OpenAI()
MODEL = 'gpt-4o-mini'

In [ ]:
# Load Amazon Appliances product listings from HuggingFace.

dataset = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_meta_Appliances',
    split='full',
    trust_remote_code=True
)

print(f'Total products data: {len(dataset):,}')

In [ ]:
# Clean data, product must have a price between $1-$1000, and an elaborate description

def clean(item):
    try:
        price = float(item.get("price"))
    except (TypeError, ValueError):
        return None

    if not 1 <= price <= 1000:
        return None

    text = " ".join([
        item.get("title", "").strip(),
        *item.get("features", []),
        *item.get("description", [])
    ]).strip()

    if len(text) < 50:
        return None

    return {
        "text": text[:1000],
        "price": price
    }

items = [clean(d) for d in dataset]
items = [i for i in items if i]
print(f'Clean products data: {len(items):,}')

In [ ]:
# creates a 50-item test set and outputs the average price guess

random.seed(42)
sample_products = random.sample(items, min(50, len(items)))

prices = [i['price'] for i in sample_products]
avg_price = sum(prices) / len(prices)
print(f'Sample size: {len(sample_products)}')
print(f'Average price: ${avg_price:.2f}')
print(f'Min: ${min(prices):.2f}  Max: ${max(prices):.2f}')

In [ ]:
# create mean_absolute_error

def mean_absolute_error(predictions, actuals):
    return sum(abs(p - a) for p, a in zip(predictions, actuals)) / len(actuals)

baseline_preds = [avg_price] * len(sample_products)
actuals = [i['price'] for i in sample_products]
baseline_mae = mean_absolute_error(baseline_preds, actuals)
print(f'Always guess mean_absolute: ${baseline_mae:.2f}')

In [ ]:
#  gpt-4o-mini predictor

SYSTEM = """
    You estimate the typical retail price of Amazon products.

    Given a product description, predict the price in EUR.

    Output format:
    A single number only.

    Rules:
    - No dollar sign
    - No text
    - No explanation
    - No extra characters

    Example:
    24.99
    """

def predictor(text):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Product: {text}\n\nPrice:"},
        ],
        max_tokens=10,
    )
    raw = response.choices[0].message.content.strip()
    match = re.search(r'[\d.]+', raw)
    return float(match.group()) if match else avg_price

print('Running LLM predictions...')
llm_preds = [predictor(i['text']) for i in tqdm(sample_products)]
llm_mae = mean_absolute_error(llm_preds, actuals)
print(f'LLM mean_absolute_error: ${llm_mae:.2f}')

In [ ]:
# Compare results

print('=' * 40)
print(f'  Baseline mean abs error : ${baseline_mae:.2f}')
print(f'  LLM mean abs error : ${llm_mae:.2f}')
improvement = ((baseline_mae - llm_mae) / baseline_mae) * 100
print(f'  Improvement  : {improvement:.1f}%')
print('=' * 40)

print('\nSample predictions:')
print(f'{"Actual":>10}  {"LLM Guess":>10}  {"Error":>10}')
for actual, pred in zip(actuals[:10], llm_preds[:10]):
    print(f'${actual:>9.2f}  ${pred:>9.2f}  ${abs(actual-pred):>9.2f}')

In [ ]:
# Gradio UI

def gradio_predict(description):
    if not description.strip():
        return "⚠️ Please enter a product description."

    price = predictor(description)
    return f"💰 My guess: **${price:.2f}**"

examples = [
    ["Stainless steel electric kettle with auto shut-off and 1.7L capacity"],
    ["Wireless noise cancelling over-ear Bluetooth headphones with 30 hour battery life"],
    ["Compact air fryer with digital touchscreen and 5 quart capacity"],
    ["USB-C fast charging cable 6ft braided for iPhone and Android"],
    ["Robot vacuum cleaner with smart mapping and WiFi control"]
]

gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(
        lines=5,
        label="🛍️ Product Description",
        placeholder="Paste an Amazon product description here..."
    ),
    outputs=gr.Markdown(label="🤖 GPT-4o-mini's Price Guess"),
    examples=examples,
    title="🛒 Is The Price Right?",
    description="""
Paste a product description and see if **GPT-4o-mini** can guess its price.
""",
).launch()